In [ ]:
"""
Curved-path obstacles demo.
Obstacles follow smooth parametric (orbiting) trajectories with slight noise.
Works in Jupyter and plain Python.
"""

import math
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation

# Notebook detection
try:
    from IPython.display import HTML, display
    IN_NOTEBOOK = True
except Exception:
    IN_NOTEBOOK = False

random.seed(0)
np.random.seed(0)

class DynamicEnvCurved:
    def __init__(self, size=60, dt=1.0):
        self.size = int(size)
        self.dt = float(dt)
        self.goal = (self.size - 1, self.size - 1)
        self.robot = (0, 0)

        # Obstacles parameters:
        # Each obstacle: [cx, cy, base_radius, curr_radius, ang, ang_vel, radius_noise_scale, ang_noise_scale]
        # cx,cy = center around which obstacle orbits (kept inside bounds)
        # base_radius = nominal orbit radius; curr_radius = current perturbed radius
        # ang = current phase angle; ang_vel = angular speed (rad/step)
        # radius_noise_scale, ang_noise_scale = small random fluctuations per step
        self.obstacles = []

        # Create several obstacles with different centers, radii, angular speeds
        centers = [
            (12, 10), (30, 8), (20, 25), (45, 15), (40, 40),
            (10, 45), (50, 30), (28, 48), (8, 22), (36, 32)
        ]
        # ensure centers are valid for the grid size
        centers = [(min(self.size-5, max(5, cx)), min(self.size-5, max(5, cy))) for cx, cy in centers]

        for i, (cx, cy) in enumerate(centers):
            base_r = 3 + (i % 4) + (i * 0.1)     # varied radii between ~3 and ~7
            ang = np.random.uniform(0, 2*math.pi)
            ang_vel = np.random.uniform(0.02, 0.08) * (1 if i % 2 == 0 else -1)  # clockwise/anticlockwise
            r_noise = 0.08 + (i % 3) * 0.03
            a_noise = 0.01 + (i % 4) * 0.005
            curr_r = base_r + np.random.normal(scale=0.2)
            self.obstacles.append([float(cx), float(cy), float(base_r), float(curr_r), float(ang), float(ang_vel), float(r_noise), float(a_noise)])

    def reset(self):
        self.robot = (0, 0)
        return self.robot

    def move_obstacles(self):
        """Update each obstacle by stepping its angle and slightly perturbing its radius/angle (smooth noise)."""
        for obs in self.obstacles:
            cx, cy, base_r, curr_r, ang, ang_vel, r_noise, a_noise = obs

            # Smooth (small) noise on angle and radius (Gaussian)
            ang += ang_vel + np.random.normal(scale=a_noise)
            # gently pull radius toward base_r while adding noise (gives gentle breathing)
            curr_r += 0.02 * (base_r - curr_r) + np.random.normal(scale=r_noise * 0.02)

            # compute new continuous position
            x = cx + curr_r * math.cos(ang)
            y = cy + curr_r * math.sin(ang)

            # ensure we don't stray out of bounds; clamp if necessary and slightly reflect phase
            if x < 0:
                x = 0; ang += math.pi * 0.1
            if x > self.size - 1:
                x = self.size - 1; ang += math.pi * 0.1
            if y < 0:
                y = 0; ang += math.pi * 0.1
            if y > self.size - 1:
                y = self.size - 1; ang += math.pi * 0.1

            # store back updated angle and radius; keep cx,cy,base_r,ang_vel,noise scales unchanged
            obs[3] = curr_r
            obs[4] = ang
            obs[0] = cx
            obs[1] = cy
            # for convenience, we will temporarily return current position via computed x,y in predict/plot

    def obstacle_position(self, obs):
        """Given an obstacle parameter list, return its current (x,y)."""
        cx, cy, base_r, curr_r, ang, ang_vel, r_noise, a_noise = obs
        x = cx + curr_r * math.cos(ang)
        y = cy + curr_r * math.sin(ang)
        return (x, y)

    def predict_obstacles(self, steps=1):
        """Predict next-step positions using a single-step parametric update (ang + ang_vel)."""
        preds = []
        for obs in self.obstacles:
            cx, cy, base_r, curr_r, ang, ang_vel, r_noise, a_noise = obs
            # predict angle next step (without extra noise) for short-term forecast
            ang_next = ang + ang_vel
            # predict a slight relaxation of radius toward base radius (no random term for deterministic prediction)
            curr_r_next = curr_r + 0.02 * (base_r - curr_r)
            px = cx + curr_r_next * math.cos(ang_next)
            py = cy + curr_r_next * math.sin(ang_next)
            preds.append((px, py))
        return preds

    def step(self, action):
        # robot grid move (4-neighbor)
        x, y = self.robot
        if action == 0: x += 1
        elif action == 1: x -= 1
        elif action == 2: y += 1
        elif action == 3: y -= 1

        x = int(np.clip(x, 0, self.size - 1))
        y = int(np.clip(y, 0, self.size - 1))
        self.robot = (x, y)

        # advance obstacles along their smooth curved paths
        self.move_obstacles()
        # compute predicted positions
        predicted = self.predict_obstacles()

        # collision check: round obstacle positions to nearest grid cell
        for obs in self.obstacles:
            ox, oy = self.obstacle_position(obs)
            if (x, y) == (int(round(ox)), int(round(oy))):
                return self.robot, -100, True

        # avoid predicted collisions (round)
        for px, py in predicted:
            if (x, y) == (int(round(px)), int(round(py))):
                return self.robot, -10, False

        # goal
        if self.robot == self.goal:
            return self.robot, 100, True

        return self.robot, -1, False

# Simple greedy policy with prediction avoidance still
def choose_action(env):
    rx, ry = env.robot
    gx, gy = env.goal
    dx = gx - rx
    dy = gy - ry

    if abs(dx) >= abs(dy):
        action = 0 if dx > 0 else 1 if dx < 0 else (2 if dy > 0 else 3)
    else:
        action = 2 if dy > 0 else 3 if dy < 0 else (0 if dx > 0 else 1)

    # build unsafe set from predicted obstacle grid cells (and neighbors)
    preds = env.predict_obstacles()
    unsafe = set()
    for px, py in preds:
        cx = int(round(px))
        cy = int(round(py))
        for nx in (cx-1, cx, cx+1):
            for ny in (cy-1, cy, cy+1):
                if 0 <= nx < env.size and 0 <= ny < env.size:
                    unsafe.add((nx, ny))

    def dest_cell(a):
        x, y = rx, ry
        if a == 0: x += 1
        if a == 1: x -= 1
        if a == 2: y += 1
        if a == 3: y -= 1
        x = int(np.clip(x, 0, env.size - 1))
        y = int(np.clip(y, 0, env.size - 1))
        return (x, y)

    intended = dest_cell(action)
    if intended in unsafe:
        # try alternatives sorted by Manhattan distance to goal
        alts = [a for a in (0,1,2,3) if a != action]
        scored = []
        for a in alts:
            c = dest_cell(a)
            score = float('inf') if c in unsafe else abs(env.goal[0] - c[0]) + abs(env.goal[1] - c[1])
            scored.append((score, a))
        scored.sort()
        action = scored[0][1]

    return action

# Animation runner
def run_simulation(size=60, frames=1000, interval=80):
    env = DynamicEnvCurved(size=size)

    fig, ax = plt.subplots(figsize=(8,8))
    ax.set_xlim(-1, env.size)
    ax.set_ylim(-1, env.size)
    ax.set_xticks(np.arange(0, env.size, max(1, env.size//10)))
    ax.set_yticks(np.arange(0, env.size, max(1, env.size//10)))
    ax.grid(True)

    robot_plot, = ax.plot([], [], 'bo', markersize=5, label='Robot')
    goal_plot, = ax.plot(env.goal[0], env.goal[1], 'go', markersize=8, label='Goal')

    obs_plots = [ax.plot([], [], marker='s', linestyle='None', markersize=6, label=f'Obs{i}')[0]
                 for i in range(len(env.obstacles))]
    pred_plots = [ax.plot([], [], marker='x', linestyle='None', markersize=5)[0] for _ in env.obstacles]

    ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1.0))

    def init():
        robot_plot.set_data([], [])
        for op in obs_plots:
            op.set_data([], [])
        for pp in pred_plots:
            pp.set_data([], [])
        return [robot_plot] + obs_plots + pred_plots

    def update(frame):
        action = choose_action(env)
        pos, reward, done = env.step(action)
        robot_plot.set_data([pos[0]], [pos[1]])

        # plot obstacles at their parametric positions
        for i, obs in enumerate(env.obstacles):
            ox, oy = env.obstacle_position(obs)
            obs_plots[i].set_data([ox], [oy])

        # predicted next step positions
        preds = env.predict_obstacles()
        for i, (px, py) in enumerate(preds):
            pred_plots[i].set_data([px], [py])

        return [robot_plot] + obs_plots + pred_plots

    ani = animation.FuncAnimation(fig, update, init_func=init, frames=frames, interval=interval, blit=True)

    if IN_NOTEBOOK:
        try:
            display(HTML(ani.to_jshtml()))
        except Exception:
            plt.show()
    else:
        plt.show()

if __name__ == "__main__":
    # run with a large grid so robot path looks curved
    run_simulation(size=60, frames=1200, interval=60)


 ROBOTICS PATH PLANNING BENCHMARK ENGINE
\nInitializing Deep Q-Network training loop (500 Episodes)...
 Episode  100 | Epsilon: 0.050 | rolling Avg Reward: -174.8 | Success Rate: 0.0%
 Episode  200 | Epsilon: 0.050 | rolling Avg Reward: -181.8 | Success Rate: 2.0%
 Episode  300 | Epsilon: 0.050 | rolling Avg Reward: -180.5 | Success Rate: 5.0%
 Episode  400 | Epsilon: 0.050 | rolling Avg Reward: -173.8 | Success Rate: 11.0%
 Episode  500 | Epsilon: 0.050 | rolling Avg Reward: -185.9 | Success Rate: 1.0%
DQN Training Complete. Optimizing final target weights...\n
Evaluating planners over 50 random testing episodes with mixed obstacle vectors...
\n================================================================================
NAVIGATION PLANNER     | SUC%     | CRASH%   | AVG STEPS  | LATENCY (ms) | PATH QUALITY   
Dijkstra               |    0.0% |  100.0% |       0.0  |       0.491 | N/A            
Reactive A*            |    0.0% |  100.0% |       0.0  |       1.291 | N/A           